In [8]:
"""
Logistics Database — Data Quality Audit
=========================================

Runs locally against your own CSV files. No data leaves your machine —
only the printed summary report is meant to be shared back.

HOW TO USE
----------
1. Put this script in a folder alongside (or pointing to) your 14 CSV files.
2. Edit the CONFIG section below:
   - FOLDER: path to your CSVs
   - FILES: map each table name to its actual filename
   - Column names inside each check: adjust to match your real headers
     (this schema was described to me, not the actual column names, so
     these are best-guess defaults you should double check).
3. Run:  python audit.py
4. Paste the printed report back — that's all I need to tell you what
   actually needs cleaning vs what's already solid.

WHAT IT CHECKS
--------------
1. Structural   — shape, dtypes, % nulls per column, duplicate PK count
2. FK integrity — orphaned foreign keys across the 14-table schema
3. Range/logic  — negative values, impossible ratios, delivery-before-pickup
4. Date coverage— continuity across the 3-year span, partial trailing months
"""

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

# ============================================================
# CONFIG — edit this section to match your actual files/columns
# ============================================================

FOLDER = Path(r"C:\Projects\3Year Truck Operations\data")  # <-- change to your CSV folder

FILES = {
    "drivers": "drivers.csv",
    "trucks": "trucks.csv",
    "trailers": "trailers.csv",
    "customers": "customers.csv",
    "facilities": "facilities.csv",
    "routes": "routes.csv",
    "loads": "loads.csv",
    "trips": "trips.csv",
    "fuel_purchases": "fuel_purchases.csv",
    "maintenance_records": "maintenance_records.csv",
    "delivery_events": "delivery_events.csv",
    "safety_incidents": "safety_incidents.csv",
    "driver_monthly_metrics": "driver_monthly_metrics.csv",
    "truck_utilization_metrics": "truck_utilization_metrics.csv",
}

PRIMARY_KEYS = {
    "drivers": "driver_id",
    "trucks": "truck_id",
    "trailers": "trailer_id",
    "customers": "customer_id",
    "facilities": "facility_id",
    "routes": "route_id",
    "loads": "load_id",
    "trips": "trip_id",
    "fuel_purchases": "fuel_purchase_id",
    "maintenance_records": "maintenance_id",
    "delivery_events": "event_id",
    "safety_incidents": "incident_id",
}

# (child_table, fk_column, parent_table, parent_pk)
FOREIGN_KEYS = [
    ("loads", "customer_id", "customers", "customer_id"),
    ("loads", "route_id", "routes", "route_id"),
    ("trips", "load_id", "loads", "load_id"),
    ("trips", "driver_id", "drivers", "driver_id"),
    ("trips", "truck_id", "trucks", "truck_id"),
    ("trips", "trailer_id", "trailers", "trailer_id"),
    ("fuel_purchases", "trip_id", "trips", "trip_id"),
    ("fuel_purchases", "truck_id", "trucks", "truck_id"),
    ("fuel_purchases", "driver_id", "drivers", "driver_id"),
    ("maintenance_records", "truck_id", "trucks", "truck_id"),
    ("delivery_events", "load_id", "loads", "load_id"),
    ("delivery_events", "trip_id", "trips", "trip_id"),
    ("delivery_events", "facility_id", "facilities", "facility_id"),
    ("safety_incidents", "trip_id", "trips", "trip_id"),
    ("safety_incidents", "truck_id", "trucks", "truck_id"),
    ("safety_incidents", "driver_id", "drivers", "driver_id"),
]

# ============================================================
# LOADING
# ============================================================

def load_tables():
    tables = {}
    for name, filename in FILES.items():
        path = FOLDER / filename
        if not path.exists():
            print(f"  [SKIP] {name}: file not found at {path}")
            continue
        try:
            tables[name] = pd.read_csv(path, low_memory=False)
        except Exception as e:
            print(f"  [ERROR] {name}: failed to read — {e}")
    return tables


# ============================================================
# 1. STRUCTURAL CHECK
# ============================================================

def structural_check(tables):
    print("\n" + "=" * 70)
    print("1. STRUCTURAL CHECK — shape, dtypes, nulls, duplicate PKs")
    print("=" * 70)
    for name, df in tables.items():
        print(f"\n--- {name} ---")
        print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

        null_pct = (df.isna().mean() * 100).round(2)
        nulls_present = null_pct[null_pct > 0].sort_values(ascending=False)
        if len(nulls_present):
            print("Nulls (%):")
            print(nulls_present.to_string())
        else:
            print("Nulls: none")

        pk = PRIMARY_KEYS.get(name)
        if pk and pk in df.columns:
            dupes = df[pk].duplicated().sum()
            print(f"Duplicate PK ({pk}): {dupes:,}")
        elif pk:
            print(f"[WARNING] Expected PK column '{pk}' not found in {name}")

        full_dupes = df.duplicated().sum()
        if full_dupes:
            print(f"Fully duplicate rows: {full_dupes:,}")

        # dtype sanity: object columns that look numeric or date-like
        for col in df.select_dtypes(include="object").columns:
            sample = df[col].dropna().astype(str).head(20)
            if sample.str.match(r"^\d{4}-\d{2}-\d{2}").any():
                print(f"[CHECK] '{col}' looks like a date but is stored as text — consider parsing to datetime")


# ============================================================
# 2. FOREIGN KEY INTEGRITY
# ============================================================

def fk_check(tables):
    print("\n" + "=" * 70)
    print("2. FOREIGN KEY INTEGRITY — orphaned references")
    print("=" * 70)
    for child, fk_col, parent, parent_pk in FOREIGN_KEYS:
        if child not in tables or parent not in tables:
            continue
        cdf, pdf = tables[child], tables[parent]
        if fk_col not in cdf.columns or parent_pk not in pdf.columns:
            print(f"[WARNING] Skipping {child}.{fk_col} -> {parent}.{parent_pk} — column missing")
            continue

        child_keys = cdf[fk_col].dropna()
        parent_keys = set(pdf[parent_pk].dropna())
        orphans = ~child_keys.isin(parent_keys)
        orphan_count = orphans.sum()

        status = "OK" if orphan_count == 0 else "ISSUE"
        print(f"[{status}] {child}.{fk_col} -> {parent}.{parent_pk}: "
              f"{orphan_count:,} orphaned rows out of {len(child_keys):,}")


# ============================================================
# 3. RANGE / LOGIC CHECKS
# ============================================================

def safe_col(df, *candidates):
    """Return the first matching column name found in df, or None."""
    for c in candidates:
        if c in df.columns:
            return c
    return None


def range_checks(tables):
    print("\n" + "=" * 70)
    print("3. RANGE / LOGIC CHECKS")
    print("=" * 70)

    # --- Routes: distance should be positive ---
    if "routes" in tables:
        df = tables["routes"]
        col = safe_col(df, "typical_distance_miles", "distance", "distance_miles", "miles")
        if col:
            bad = (df[col] <= 0).sum()
            print(f"[routes] '{col}' <= 0: {bad:,} rows")
        rate_col = safe_col(df, "base_rate_per_mile")
        if rate_col:
            bad = (df[rate_col] <= 0).sum()
            print(f"[routes] '{rate_col}' <= 0: {bad:,} rows")

    # --- Loads: revenue should be positive ---
    if "loads" in tables:
        df = tables["loads"]
        col = safe_col(df, "revenue", "load_revenue", "rate")
        if col:
            bad = (df[col] <= 0).sum()
            print(f"[loads] '{col}' <= 0: {bad:,} rows")

    # --- Trips: fuel consumption / duration should be non-negative ---
    if "trips" in tables:
        df = tables["trips"]
        for cand in [("fuel_gallons_used", "fuel_consumption", "fuel_used"),
                     ("actual_duration_hours", "duration", "trip_duration"),
                     ("actual_distance_miles",),
                     ("idle_time_hours",)]:
            col = safe_col(df, *cand)
            if col:
                bad = (df[col] < 0).sum()
                print(f"[trips] '{col}' < 0: {bad:,} rows")

        # average_mpg is precomputed here — range-check it directly
        mpg_col = safe_col(df, "average_mpg", "mpg")
        if mpg_col:
            implausible = ((df[mpg_col] < 2) | (df[mpg_col] > 15)).sum()
            print(f"[trips] Implausible '{mpg_col}' (<2 or >15): {implausible:,} rows")

        # Explain the nulls in driver_id/truck_id/trailer_id via trip_status
        status_col = safe_col(df, "trip_status", "status")
        if status_col:
            for fk in ["driver_id", "truck_id", "trailer_id"]:
                if fk in df.columns:
                    null_rows = df[df[fk].isna()]
                    if len(null_rows):
                        print(f"[trips] '{fk}' nulls by '{status_col}':")
                        print(null_rows[status_col].value_counts().to_string())

    # --- Fuel purchases: gallons and price should be positive ---
    if "fuel_purchases" in tables:
        df = tables["fuel_purchases"]
        for cand in [("gallons", "fuel_gallons", "quantity"),
                     ("price", "price_per_gallon", "unit_price"),
                     ("total_cost", "cost", "amount")]:
            col = safe_col(df, *cand)
            if col:
                bad = (df[col] <= 0).sum()
                print(f"[fuel_purchases] '{col}' <= 0: {bad:,} rows")

        # MPG plausibility if both odometer/miles and gallons exist
        miles_col = safe_col(df, "miles", "distance")
        gal_col = safe_col(df, "gallons", "fuel_gallons", "quantity")
        if miles_col and gal_col:
            mpg = df[miles_col] / df[gal_col].replace(0, np.nan)
            implausible = ((mpg < 2) | (mpg > 15)).sum()
            print(f"[fuel_purchases] Implausible MPG (<2 or >15): {implausible:,} rows")

    # --- Maintenance: cost and downtime should be non-negative ---
    if "maintenance_records" in tables:
        df = tables["maintenance_records"]
        for cand in [("cost", "maintenance_cost", "repair_cost"),
                     ("downtime", "downtime_hours", "downtime_days")]:
            col = safe_col(df, *cand)
            if col:
                bad = (df[col] < 0).sum()
                print(f"[maintenance_records] '{col}' < 0: {bad:,} rows")

    # --- Delivery events: on_time_flag consistency, detention, event types ---
    if "delivery_events" in tables:
        df = tables["delivery_events"]
        sched_col = safe_col(df, "scheduled_datetime")
        actual_col = safe_col(df, "actual_datetime")
        flag_col = safe_col(df, "on_time_flag")
        detention_col = safe_col(df, "detention_minutes")

        if detention_col:
            bad = (df[detention_col] < 0).sum()
            print(f"[delivery_events] '{detention_col}' < 0: {bad:,} rows")

        if sched_col and actual_col:
            sched = pd.to_datetime(df[sched_col], errors="coerce")
            actual = pd.to_datetime(df[actual_col], errors="coerce")
            unparseable = ((sched.isna() & df[sched_col].notna())
                            | (actual.isna() & df[actual_col].notna())).sum()
            if unparseable:
                print(f"[delivery_events] Unparseable scheduled/actual datetimes: {unparseable:,} rows")

            if flag_col:
                computed_on_time = (actual <= sched)
                stated_on_time = df[flag_col].astype(bool)
                mismatch = (computed_on_time != stated_on_time) & sched.notna() & actual.notna()
                print(f"[delivery_events] '{flag_col}' disagrees with actual<=scheduled: "
                      f"{mismatch.sum():,} rows")

        if "event_type" in df.columns:
            print(f"[delivery_events] event_type breakdown:")
            print(df["event_type"].value_counts().to_string())

    # --- Safety incidents: cost fields non-negative ---
    if "safety_incidents" in tables:
        df = tables["safety_incidents"]
        for cand in [("vehicle_damage_cost",), ("cargo_damage_cost",), ("claim_amount",)]:
            col = safe_col(df, *cand)
            if col:
                bad = (df[col] < 0).sum()
                print(f"[safety_incidents] '{col}' < 0: {bad:,} rows")
        # flags should be boolean-like (0/1 or True/False)
        for flag_col in ["at_fault_flag", "injury_flag", "preventable_flag"]:
            if flag_col in df.columns:
                unexpected = ~df[flag_col].isin([0, 1, True, False, np.nan])
                if unexpected.sum():
                    print(f"[safety_incidents] '{flag_col}' has unexpected values: "
                          f"{df.loc[unexpected, flag_col].unique()}")


# ============================================================
# 4. DATE COVERAGE CHECK
# ============================================================

def date_coverage_check(tables):
    print("\n" + "=" * 70)
    print("4. DATE COVERAGE CHECK — continuity across the 3-year span")
    print("=" * 70)

    # Adjust this list to whichever table/column actually carries the
    # primary transaction date for your dataset (e.g. trips start date).
    candidates = [
        ("trips", ["dispatch_date", "trip_start", "start_date", "trip_date"]),
        ("loads", ["load_date", "booking_date", "created_date"]),
        ("fuel_purchases", ["purchase_date"]),
        ("maintenance_records", ["maintenance_date"]),
        ("safety_incidents", ["incident_date"]),
        ("delivery_events", ["scheduled_datetime"]),
    ]

    for table, cols in candidates:
        if table not in tables:
            continue
        df = tables[table]
        col = safe_col(df, *cols)
        if not col:
            print(f"[{table}] [SKIP] no recognizable date column found — "
                  f"update date_coverage_check() candidates to match your headers")
            continue

        dates = pd.to_datetime(df[col], errors="coerce")
        bad_dates = dates.isna().sum() - df[col].isna().sum()  # unparseable but non-null
        print(f"[{table}] Date range: {dates.min()} to {dates.max()}")
        if bad_dates > 0:
            print(f"[{table}] Unparseable date values: {bad_dates:,}")

        monthly_counts = dates.dt.to_period("M").value_counts().sort_index()
        print(f"[{table}] Months covered: {len(monthly_counts)}")
        print(f"[{table}] Min/Max monthly row count: "
              f"{monthly_counts.min():,} / {monthly_counts.max():,}")

        # Flag first/last month if they look like partial periods
        # (row count less than half the median — same pattern found in DataCo)
        median_count = monthly_counts.median()
        if monthly_counts.iloc[0] < median_count * 0.5:
            print(f"[{table}] [FLAG] First month ({monthly_counts.index[0]}) looks partial "
                  f"({monthly_counts.iloc[0]:,} rows vs median {median_count:,.0f})")
        if monthly_counts.iloc[-1] < median_count * 0.5:
            print(f"[{table}] [FLAG] Last month ({monthly_counts.index[-1]}) looks partial "
                  f"({monthly_counts.iloc[-1]:,} rows vs median {median_count:,.0f})")

        # Gap check — any month with zero rows in between min and max
        full_range = pd.period_range(dates.min().to_period("M"), dates.max().to_period("M"), freq="M")
        missing_months = [str(m) for m in full_range if m not in monthly_counts.index]
        if missing_months:
            print(f"[{table}] [FLAG] Months with zero rows: {missing_months}")


# ============================================================
# MAIN
# ============================================================

def main():
    print("Loading tables from:", FOLDER.resolve())
    tables = load_tables()
    print(f"\nLoaded {len(tables)}/{len(FILES)} tables successfully.")

    structural_check(tables)
    fk_check(tables)
    range_checks(tables)
    date_coverage_check(tables)

    print("\n" + "=" * 70)
    print("DONE — paste this full output back for a cleaning game plan.")
    print("=" * 70)


if __name__ == "__main__":
    main()

Loading tables from: C:\Projects\3Year Truck Operations\data

Loaded 14/14 tables successfully.

1. STRUCTURAL CHECK — shape, dtypes, nulls, duplicate PKs

--- drivers ---
Shape: 150 rows x 12 columns
Nulls (%):
termination_date    82.67
Duplicate PK (driver_id): 0
[CHECK] 'hire_date' looks like a date but is stored as text — consider parsing to datetime
[CHECK] 'termination_date' looks like a date but is stored as text — consider parsing to datetime
[CHECK] 'date_of_birth' looks like a date but is stored as text — consider parsing to datetime

--- trucks ---
Shape: 120 rows x 11 columns
Nulls: none
Duplicate PK (truck_id): 0
[CHECK] 'acquisition_date' looks like a date but is stored as text — consider parsing to datetime

--- trailers ---
Shape: 180 rows x 9 columns
Nulls: none
Duplicate PK (trailer_id): 0
[CHECK] 'acquisition_date' looks like a date but is stored as text — consider parsing to datetime

--- customers ---
Shape: 200 rows x 8 columns
Nulls: none
Duplicate PK (customer_i